# LLaMA-3-8B QLoRA for Sinhala offensive-language classification

Fine-tune `meta-llama/Meta-Llama-3-8B` with a sequence-classification head and QLoRA, then evaluate the same trained adapter on aligned clean, light-transliterated, and heavy-transliterated SOLD test posts.

Default: train on heavy transliteration. Change `TRAIN_CONDITIONS` to run clean/light/heavy experiments sequentially. No Hugging Face token is stored in this notebook.

## 1. Install dependencies

Run this cell once and restart the kernel before continuing.

In [ ]:
%pip install -q -U "transformers>=4.45" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.46" "datasets>=3.0" pandas scikit-learn matplotlib seaborn

## 2. Configuration

In [ ]:
from pathlib import Path
import gc
import json
import os
import platform
import time

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_ID = "meta-llama/Meta-Llama-3-8B"
DATA_ROOT = None
OUTPUT_ROOT = Path("llama/lora_results")

# Each condition is a separate training run; every run is tested on all three conditions.
TRAIN_CONDITIONS = ["heavy"]  # use ["clean", "light", "heavy"] for the full matrix
TRAIN_SUBSET = None           # e.g. 200 for a smoke test
EPOCHS = 3.0
LEARNING_RATE = 1e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
MAX_LENGTH = 128
LORA_R = 16
VALIDATION_FRACTION = 0.10

LABEL_NAME = {0: "NOT", 1: "OFF"}
np.random.seed(SEED)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA training requires a CUDA GPU.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"cuda:{i} {props.name}, {props.total_memory / 2**30:.1f} GiB")

## 3. Authenticate and locate data

Accept the gated LLaMA licence first. Set `HF_TOKEN` as an environment/Kaggle secret or authenticate interactively; never paste a token into the notebook.

In [ ]:
from huggingface_hub import login


hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated from HF_TOKEN.")
else:
    print("Using cached Hugging Face credentials; call login() interactively if required.")

REQUIRED = [
    Path("sold/train.jsonl"), Path("sold/test.jsonl"),
    Path("outputs/train/codemix_light.jsonl"),
    Path("outputs/train/codemix_heavy.jsonl"),
    Path("outputs/test/codemix_light.jsonl"),
    Path("outputs/test/codemix_heavy.jsonl"),
]

def is_data_root(path):
    path = Path(path)
    return all((path / rel).is_file() for rel in REQUIRED)

def find_data_root(explicit=None):
    if explicit is not None:
        explicit = Path(explicit)
        if not is_data_root(explicit):
            raise FileNotFoundError(f"Invalid DATA_ROOT: {explicit.resolve()}")
        return explicit
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent / "data", parent):
            if is_data_root(candidate):
                return candidate
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for train_file in kaggle.glob("**/sold/train.jsonl"):
            candidate = train_file.parent.parent
            if is_data_root(candidate):
                return candidate
    raise FileNotFoundError("Could not locate the dataset. Set DATA_ROOT explicitly.")

DATA_ROOT = find_data_root(DATA_ROOT)
FILES = {
    "clean_train": DATA_ROOT / "sold/train.jsonl",
    "clean_test": DATA_ROOT / "sold/test.jsonl",
    "light_train": DATA_ROOT / "outputs/train/codemix_light.jsonl",
    "heavy_train": DATA_ROOT / "outputs/train/codemix_heavy.jsonl",
    "light_test": DATA_ROOT / "outputs/test/codemix_light.jsonl",
    "heavy_test": DATA_ROOT / "outputs/test/codemix_heavy.jsonl",
}
print("DATA_ROOT =", DATA_ROOT.resolve())

## 4. Load and validate aligned datasets

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return pd.DataFrame(json.loads(line) for line in handle if line.strip())

def canonicalize(df, condition, split):
    id_candidates = ("post_id", "source_id", "id", "sentence_id", "orig_id")
    text_candidates = ("text", "noisy_text", "transliterated", "transliteration", "romanized")
    id_col = next((c for c in id_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)
    if id_col is None or text_col is None or "label" not in df.columns:
        raise KeyError(f"{condition}_{split}: unsupported columns {df.columns.tolist()}")
    out = df[[id_col, text_col, "label"]].rename(columns={id_col: "id", text_col: "text"}).copy()
    out["id"] = out["id"].astype(str)
    out["text"] = out["text"].astype(str)
    if out["label"].dtype == object:
        out["label"] = out["label"].astype(str).str.strip().map(
            {"NOT": 0, "OFF": 1, "not": 0, "off": 1, "not-offensive": 0, "offensive": 1}
        )
    if out["label"].isna().any():
        raise ValueError(f"{condition}_{split}: unmapped labels")
    out["label"] = out["label"].astype(int)
    if out["id"].duplicated().any() or out["text"].str.strip().eq("").any():
        raise ValueError(f"{condition}_{split}: duplicate IDs or empty text")
    if not set(out["label"]).issubset({0, 1}):
        raise ValueError(f"{condition}_{split}: labels must be 0/1")
    out["condition"], out["split"] = condition, split
    return out

frames = {}
for condition in ("clean", "light", "heavy"):
    for split in ("train", "test"):
        key = f"{condition}_{split}"
        frames[key] = canonicalize(read_jsonl(FILES[key]), condition, split)
        print(f"{key:12s}: n={len(frames[key])}, labels={frames[key]['label'].value_counts().sort_index().to_dict()}")

for split in ("train", "test"):
    reference = frames[f"clean_{split}"].set_index("id")
    for condition in ("light", "heavy"):
        candidate = frames[f"{condition}_{split}"].set_index("id")
        assert set(candidate.index) == set(reference.index), f"{condition}_{split}: ID mismatch"
        candidate = candidate.loc[reference.index]
        assert np.array_equal(candidate["label"], reference["label"]), f"{condition}_{split}: label mismatch"

display(pd.DataFrame({
    "clean": frames["clean_test"]["text"].head(3),
    "light": frames["light_test"]["text"].head(3),
    "heavy": frames["heavy_test"]["text"].head(3),
    "label": frames["clean_test"]["label"].head(3).map(LABEL_NAME),
}))

## 5. Tokenizer, QLoRA model, and datasets

In [ ]:
from datasets import ClassLabel, Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig,
    DataCollatorWithPadding, set_seed,
)

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def build_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def make_dataset(frame, tokenizer):
    dataset = Dataset.from_pandas(frame[["id", "text", "label"]], preserve_index=False)
    dataset = dataset.cast_column("label", ClassLabel(names=["NOT", "OFF"]))
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )

def build_model(tokenizer):
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        id2label=LABEL_NAME,
        label2id={"NOT": 0, "OFF": 1},
        pad_token_id=tokenizer.pad_token_id,
        quantization_config=quantization,
        device_map="auto",
        dtype=COMPUTE_DTYPE,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        task_type="SEQ_CLS",
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"],
    ))
    model.print_trainable_parameters()
    return model

## 6. Train and evaluate

Macro-F1 is the primary metric. Class-weighted loss compensates for SOLD imbalance. The validation split is drawn only from the selected training condition; test data is never used for checkpoint selection.

In [ ]:
import dataclasses
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    f1_score, precision_recall_fscore_support,
)
from transformers import Trainer, TrainingArguments

def metric_values(labels, predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "not_precision": precision[0], "not_recall": recall[0], "not_f1": f1[0],
        "off_precision": precision[1], "off_recall": recall[1], "off_f1": f1[1],
    }

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    return metric_values(labels, np.argmax(logits, axis=-1))

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device, outputs.logits.dtype)
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def training_arguments(output_dir):
    kwargs = dict(
        output_dir=str(output_dir / "checkpoints"),
        seed=SEED,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        eval_accumulation_steps=8,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.06,
        weight_decay=0.01,
        max_grad_norm=0.3,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        dataloader_num_workers=2,
        dataloader_pin_memory=False,
        group_by_length=True,
        label_names=["labels"],
    )
    valid = {field.name for field in dataclasses.fields(TrainingArguments)}
    return TrainingArguments(**{key: value for key, value in kwargs.items() if key in valid})

def run_condition(condition):
    set_seed(SEED)
    output_dir = OUTPUT_ROOT / f"train_{condition}_seed{SEED}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_frame = frames[f"{condition}_train"].copy()
    if TRAIN_SUBSET is not None:
        train_frame = train_frame.sample(
            n=min(TRAIN_SUBSET, len(train_frame)), random_state=SEED
        ).reset_index(drop=True)

    tokenizer = build_tokenizer()
    full_train = make_dataset(train_frame, tokenizer)
    split = full_train.train_test_split(
        test_size=VALIDATION_FRACTION, seed=SEED, stratify_by_column="label"
    )
    train_ds, validation_ds = split["train"], split["test"]
    counts = np.bincount(np.asarray(train_ds["label"]), minlength=2)
    class_weights = torch.tensor(
        counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float32
    )
    print(f"{condition}: train={len(train_ds)}, validation={len(validation_ds)}, counts={counts.tolist()}")

    model = build_model(tokenizer)
    trainer = WeightedTrainer(
        model=model,
        args=training_arguments(output_dir),
        train_dataset=train_ds,
        eval_dataset=validation_ds,
        data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )
    train_result = trainer.train()
    trainer.model.save_pretrained(output_dir / "adapter")
    tokenizer.save_pretrained(output_dir / "adapter")

    result_rows, prediction_frames = [], []
    for test_condition in ("clean", "light", "heavy"):
        test_frame = frames[f"{test_condition}_test"].copy()
        test_ds = make_dataset(test_frame, tokenizer)
        output = trainer.predict(test_ds)
        predictions = np.argmax(output.predictions, axis=-1)
        metrics = metric_values(np.asarray(test_ds["label"]), predictions)
        result_rows.append({
            "train_condition": condition,
            "test_condition": test_condition,
            "n": len(test_ds),
            **metrics,
        })
        prediction_frames.append(pd.DataFrame({
            "id": test_frame["id"],
            "train_condition": condition,
            "test_condition": test_condition,
            "label": test_frame["label"],
            "prediction": predictions,
        }))
        print(f"\n=== train={condition}, test={test_condition} ===")
        print(classification_report(
            test_frame["label"], predictions,
            labels=[0, 1], target_names=["NOT", "OFF"], digits=4, zero_division=0,
        ))

    results = pd.DataFrame(result_rows)
    predictions = pd.concat(prediction_frames, ignore_index=True)
    results.to_csv(output_dir / "metrics.csv", index=False)
    predictions.to_json(
        output_dir / "predictions.jsonl", orient="records", lines=True, force_ascii=False
    )
    with (output_dir / "run_summary.json").open("w", encoding="utf-8") as handle:
        json.dump({
            "model_id": MODEL_ID,
            "train_condition": condition,
            "seed": SEED,
            "train_rows": len(train_ds),
            "validation_rows": len(validation_ds),
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "lora_r": LORA_R,
            "max_length": MAX_LENGTH,
            "train_metrics": train_result.metrics,
            "test_metrics": result_rows,
        }, handle, indent=2, default=float)

    del trainer, model, tokenizer, train_ds, validation_ds
    gc.collect()
    torch.cuda.empty_cache()
    return results, predictions

## 7. Run configured experiments and save the robustness matrix

A full three-condition run can take many hours. Start with `TRAIN_SUBSET=200` for a smoke test before setting it back to `None`.

In [ ]:
all_results = []
all_predictions = []
started = time.time()

for condition in TRAIN_CONDITIONS:
    print("\n" + "=" * 72)
    print("TRAIN CONDITION:", condition)
    print("=" * 72)
    condition_results, condition_predictions = run_condition(condition)
    all_results.append(condition_results)
    all_predictions.append(condition_predictions)

results = pd.concat(all_results, ignore_index=True)
predictions = pd.concat(all_predictions, ignore_index=True)
results.to_csv(OUTPUT_ROOT / "all_metrics.csv", index=False)
predictions.to_json(
    OUTPUT_ROOT / "all_predictions.jsonl", orient="records", lines=True, force_ascii=False
)

matrix = results.pivot(index="train_condition", columns="test_condition", values="macro_f1")
matrix.to_csv(OUTPUT_ROOT / "macro_f1_matrix.csv")
display(results.round(4))
display(matrix.round(4))
print(f"Finished in {(time.time() - started) / 3600:.2f} hours")
print("Artifacts:", OUTPUT_ROOT.resolve())

## 8. Plot results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, max(2.5, 0.8 * len(matrix))))
sns.heatmap(matrix[["clean", "light", "heavy"]], annot=True, fmt=".4f", cmap="Blues", vmin=0, vmax=1)
plt.title("LLaMA-3-8B QLoRA macro-F1")
plt.xlabel("Test condition")
plt.ylabel("Training condition")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "macro_f1_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

## Reporting checklist

Report the base-model revision/date, LoRA targets/rank, quantization, training condition and size, validation policy, seed, epochs, learning rate, effective batch size, truncation length, and clean/light/heavy metrics. Keep smoke-test results separate from full-data results. Do not select hyperparameters using the test sets.